# ContextTranslate — Fine-tune OPUS-MT Anh → Việt (LoRA)

Đây là notebook **thử nghiệm học tập**, không phải model dịch truyện chất lượng cao đã được kiểm chứng.

- Không cần Gemini API, không tính tiền token Gemini.
- Chạy huấn luyện trên Google Colab GPU (nếu cấp được).
- Dữ liệu `novel_pairs_draft.csv` được ghép từ 2 file Word của bạn theo đoạn tương ứng, nhưng **chưa được con người duyệt kỹ** (`human_reviewed=0`). Hãy sửa lỗi trước khi dùng làm dữ liệu tốt.
- Dành ~20% ví dụ làm tập kiểm tra; với dữ liệu ít, chỉ quan sát được quy trình, không kỳ vọng dịch truyện hay.
- **OPUS-MT là model dịch máy**, không phải LLM biết theo chỉ thị ngữ cảnh. Bài này học fine-tune tiếng Anh → Việt cơ bản; giai đoạn khác sẽ bổ sung character context.

**Thao tác:** Colab → `Runtime` → `Change runtime type` → chọn GPU nếu có. Upload CSV vào `/content/`.

In [ ]:
%pip -q install "transformers>=4.48,<5" "peft>=0.14,<1" "datasets>=3.5,<5" "accelerate>=1.3,<2" "sentencepiece" "sacremoses" "pandas"

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn Runtime > Change runtime type > GPU trước khi train.")
print("GPU:",torch.cuda.get_device_name(0))

## 1. Upload dữ liệu

Upload **`novel_pairs_draft.csv`** trong thư mục của gói ZIP. Đây là ví dụ căn chỉnh sơ bộ; cần soát nội dung, nhất là các câu xưng hô. Để thử huấn luyện nhanh notebook dùng các cặp này dù `human_reviewed=0`. Không dùng để tuyên bố chất lượng model.

In [ ]:
from google.colab import files
uploaded = files.upload()  # chọn novel_pairs_draft.csv
assert 'novel_pairs_draft.csv' in uploaded, "Bạn cần upload đúng file CSV."

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("novel_pairs_draft.csv", encoding="utf-8-sig")
df = df.dropna(subset=["source_en", "target_vi"])
df["source_en"] = df["source_en"].astype(str).str.strip()
df["target_vi"] = df["target_vi"].astype(str).str.strip()
df = df[(df.source_en.str.len() >= 10) & (df.target_vi.str.len() >= 8)]
df = df.drop_duplicates(subset=["source_en", "target_vi"])
print("Số cặp dữ liệu:", len(df))
print("Số cặp được người dùng duyệt:", int(df.human_reviewed.fillna(0).astype(int).sum()))
assert len(df) >= 20, "Cần ít nhất 20 cặp cho demo này."
print(df[["source_en", "target_vi"]].head(3).to_string(index=False))
# Giữ nguyên nội dung ví dụ; không tạo câu giả định hoặc dùng bản dịch AI làm ground truth.
dataset = Dataset.from_pandas(df[["source_en", "target_vi"]].reset_index(drop=True))
sets = dataset.train_test_split(test_size=0.2, seed=42)
print("Train:", len(sets["train"]), "Eval:", len(sets["test"]))

## 2. Tải model dịch có sẵn và đo bản dịch gốc

Model OPUS-MT này yêu cầu tiền tố `>>vie<<` để chỉ ngôn ngữ đích tiếng Việt. Notebook luôn sử dụng cùng tiền tố khi học và khi dịch.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
BASE = "Helsinki-NLP/opus-mt-en-vi"
tokenizer = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE)
base_model = base_model.to("cuda")

def translate_with(model, sentence):
    model.eval()
    inputs = tokenizer(">>vie<< " + sentence, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k:v.to("cuda") for k,v in inputs.items()}
    with torch.inference_mode():
        out = model.generate(**inputs, num_beams=4, max_new_tokens=180)
    return tokenizer.decode(out[0], skip_special_tokens=True)

TEST_TEXT = "Hello, Yuuka. Where are you! Are you okay?"
before = translate_with(base_model, TEST_TEXT)
print("Trước fine-tune:", before)

## 3. Chuẩn bị token và tạo LoRA adapter

LoRA chỉ cập nhật một phần trọng số bổ sung thay vì fine-tune toàn bộ model. Bộ data hiện tại rất ít, nên đây chỉ là bài tập quan sát.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

def tokenize_batch(batch):
    texts = [">>vie<< " + t for t in batch["source_en"]]
    x = tokenizer(texts, max_length=256, truncation=True)
    labels = tokenizer(text_target=batch["target_vi"], max_length=256, truncation=True)
    x["labels"] = labels["input_ids"]
    return x

encoded = sets.map(tokenize_batch, batched=True, remove_columns=["source_en", "target_vi"])
config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)
model = get_peft_model(base_model, config)
model.print_trainable_parameters()
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

## 4. Huấn luyện

Bắt đầu chỉ với **2 epoch**. Nếu gặp thiếu bộ nhớ GPU: giảm `per_device_train_batch_size` về `1`, hoặc giảm `max_length` ở bước tokenize xuống 128.

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="/content/contexttranslate_opus_lora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    eval_strategy="epoch",
    # save_strategy="no",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=5,
    report_to="none",
    fp16=torch.cuda.is_available(),
    predict_with_generate=False
)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["test"],
    data_collator=collator,
    processing_class=tokenizer
)
trainer.train()
print('Finished training.')

## 5. So sánh và tải adapter

**Lưu ý:** LoRA đã học có thể tệ hơn model ban đầu nếu dữ liệu ít hoặc nhiễu. Cần đánh giá trên tập test chưa dùng huấn luyện, đặc biệt xưng hô Sensei–Yuuka.

In [ ]:
after = translate_with(model, TEST_TEXT)
print("EN:", TEST_TEXT)
print("Base:", before)
print("LoRA:", after)
print("Tham khảo bản trong file Word: Alo, Yuuka. em đang ở đâu! Em ổn chứ?")

ADAPTER_DIR = "/content/novel_lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Đã lưu adapter:", ADAPTER_DIR)

import shutil
shutil.make_archive("/content/novel_lora_adapter", "zip", ADAPTER_DIR)
files.download("/content/novel_lora_adapter.zip")

## 6. Đọc offline trên Windows

1. Tải file ZIP adapter và giải nén thành thư mục `novel_lora_adapter`.
2. Trong máy Windows chạy `pip install torch "transformers>=4.48,<5" "peft>=0.14,<1" sentencepiece sacremoses`.
3. Chạy `python translate_local.py` trong cùng thư mục chứa adapter. Mã này sẽ **tải model gốc một lần** từ Hugging Face. Những lần chạy sau có thể dùng cache và không cần internet, miễn file đã được tải đầy đủ.
4. Máy CPU i5-6300U sẽ chậm hơn đáng kể so với GPU. Nên dịch theo đoạn ngắn và lưu SQL Server, không dịch lại mỗi lần mở chương.

**Mở rộng sau:** tạo tập dữ liệu có `speaker`, `listener`, `scene` được xác nhận; thử LLM nhỏ hỗ trợ prompt và LoRA, vì một model dịch máy không tự hiểu hệ thống vai nói phức tạp như một LLM.